In [20]:
import requests  # fetches data from webpage
from bs4 import BeautifulSoup  # extracts data we need
import pandas as pd  # to store data as csv or excel format
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
URL = "https://sbi.co.in/web/interest-rates/deposit-rates/retail-domestic-term-deposits"

date1 = (date.today()).strftime('%d%m%Y') # Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' # Set directory path to write file

url = "https://sbi.co.in/web/interest-rates/deposit-rates/retail-domestic-term-deposits"
r = requests.get(URL)
#print(r)
soup = BeautifulSoup(r.text, 'html.parser')
table = soup.find("table", class_="table table-bordered")
rows = table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
for i in rows[2:]:  # Skip heading
    data = i.find_all("td")
    row = ['State Bank of India']
    row.extend([tr.text for tr in data])
    
    list_of_interest_rates.append(row)
print(list_of_interest_rates)
# Clean and format interest rates
list_of_interest_rates = [[sublist[i].replace('*', '').strip() + '%' if i in [3, 5] else sublist[i] for i in [1, 3, 5]] 
                          for sublist in list_of_interest_rates]

final_rates = []
tax_saver_row = []
row_count = 0
for i in list_of_interest_rates:
    x = ['State Bank of India']
    x.extend(i)  # Add tenure and rates
    x.insert(3, '')  # Placeholder for annualised_gen_rates
    x.extend([''])
    if row_count == 7:
        tax_saver_row = ['State Bank of India', 'Tax-saver Fixed Deposit', x[2], '', x[4], '']
        final_rates.append(tax_saver_row)
    final_rates.append(x)
    row_count += 1

# Create DataFrame with the new columns
pd.set_option('display.max_columns', None)
int_rate_df=pd.DataFrame(final_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate','sr_rate',
                                    'annualised_sr_rate'])
print(int_rate_df)
int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nState Bank of India = Success. Number of rows added = ", len(final_rates))

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
[['State Bank of India', '7 days to 45 days', '3.50', '3.50', '4.00', '4.00'], ['State Bank of India', '46 days to 179 days', '5.50', '5.50', '6.00', '6.00'], ['State Bank of India', '180 days to 210 days', '6.00', '6.25', '6.50', '6.75'], ['State Bank of India', '211 days to less than 1 year', '6.25', '6.50', '6.75', '7.00'], ['State Bank of India', '1 Year to less than 2 years', '6.80', '6.80', '7.30', '7.30'], ['State Bank of India', '2 years to less than 3 years', '7.00', '7.00', '7.50', '7.50'], ['State Bank of India', '3 years to less than 5 years', '6.75', '6.75', '7.25', '7.25'], ['State Bank of India', '5 years and up to 10 years', '6.50', '6.50', '7.50*', '7.50*']]
             bank_name                        tenure gen_rate  \
0  State Bank of India             7 days to 45 days    3.50%   
1  State Bank of India           46 days to 179 days    5.50%   
2  State Bank of 